# Antimicrobial Peptide Activity Prediction (E. coli) - Line-by-Line Commented

## Imports

In [ ]:
import re                                   # re = regular expressions: pattern matching on text
import numpy as np                          # numpy = fast maths on numbers; "np" is its nickname
import pandas as pd                         # pandas = tables (DataFrames), like Excel inside Python
import matplotlib.pyplot as plt             # matplotlib = drawing graphs; "plt" is its nickname
import seaborn as sns                       # seaborn = nicer-looking graphs built on matplotlib
import peptides                             # computes peptide properties (charge, hydrophobicity...)
from sklearn.model_selection import train_test_split   # splits data into training and test parts
from sklearn.svm import SVC                 # SVC = Support Vector Classifier (the SVM model)
from sklearn.ensemble import RandomForestClassifier    # Random Forest = many decision trees voting
from sklearn.naive_bayes import GaussianNB  # Gaussian Naive Bayes = probability-based model
from sklearn.pipeline import make_pipeline  # chains steps (e.g. scale, then classify) into one model
from sklearn.preprocessing import StandardScaler       # rescales each feature to a similar range
from sklearn.base import clone              # makes a fresh, untrained copy of a model
from sklearn.metrics import (accuracy_score, precision_score, recall_score,   # the 4 score functions
                             f1_score, ConfusionMatrixDisplay)                # f1 + confusion-matrix plot

sns.set_theme(style="whitegrid")            # one-time chart style: white background, light gridlines

## Configuration

In [ ]:
CSV_PATH = "dbaasp_ecoli.csv"               # name of the data file (must be in the same folder)
SEQ_COL  = "SEQUENCE"   # change to match your file   # column holding the amino-acid sequence
MIC_COL  = "MIC_µgml"  # already in ug/mL in this dataset   # column holding the MIC value
UNIT_COL = "UNIT"        # not present here -> values assumed ug/mL   # column holding units (if any)

MIC_THRESHOLD = 32           # ug/mL; active if MIC <= threshold   # the active/inactive cutoff
AA = "ACDEFGHIKLMNPQRSTVWY"                 # the 20 standard amino-acid letters
RANDOM_STATE = 42                           # fixed seed so random steps give the same result every run

## Load data

In [ ]:
df = pd.read_csv(CSV_PATH)                  # read the CSV file into a table called df
print("Columns:", list(df.columns))         # show the column names so you can check them
print("Shape:", df.shape)                   # show (number of rows, number of columns)
df.head()                                   # display the first 5 rows as a preview table

In [ ]:
required = {"sequence": SEQ_COL, "MIC": MIC_COL}   # pair each needed role with the column name you set
missing = [f"{role} -> '{col}'" for role, col in required.items() if col not in df.columns]   # list the ones NOT found in the file
if missing:                                 # an empty list counts as False, so this runs only if something is missing
    raise KeyError(                         # stop the notebook with a clear error message
        "Column(s) not found: " + "; ".join(missing)           # say which columns were not found
        + f"\nAvailable columns: {list(df.columns)}"           # show the columns that DO exist
        + "\nEdit SEQ_COL / MIC_COL in the Configuration cell to match."   # tell you how to fix it
    )
has_units = UNIT_COL in df.columns          # True if the file has a unit column, else False
print("Unit column present:", has_units)    # report that True/False answer

## Clean & filter

In [ ]:
valid_seq = re.compile(f"^[{AA}]+$")        # pattern: start (^) to end ($), only the 20 letters, one or more (+)

def first_number(x):                        # helper: pull the first number out of a value
    m = re.search(r"[-+]?\d*\.?\d+", str(x))    # look for a number (optional sign, optional decimal point)
    return float(m.group()) if m else np.nan    # return it as a decimal; if none found, return "missing" (nan)

data = df.copy()                            # work on a copy so the original df stays untouched
data[SEQ_COL] = data[SEQ_COL].astype(str).str.strip().str.upper()   # make text, trim spaces, uppercase
data = data[data[SEQ_COL].apply(lambda s: bool(valid_seq.match(s)))]   # keep only rows whose sequence matches the pattern

data["MIC_value"] = data[MIC_COL].apply(first_number)   # new column with the MIC as a clean number
data = data[data["MIC_value"].notna() & (data["MIC_value"] > 0)]   # keep rows where MIC exists and is above zero
print("Rows with valid sequence and numeric MIC:", len(data))   # report how many rows survived

## Standardise MIC units to µg/mL

In [ ]:
def norm_unit(u):                           # helper: tidy a messy unit label into a standard one
    u = str(u).strip().lower().replace("µ", "u").replace("μ", "u")   # text, trim, lowercase, turn the micro sign into "u"
    if u in {"ug/ml", "mg/l", "ug/cm3"}:    # these all mean micrograms per millilitre
        return "ug/mL"                      # return the standard mass-concentration label
    if u in {"um", "umol/l", "umol/ml"}:    # these all mean micromolar
        return "uM"                         # return the standard molar label
    return None                             # anything else is unrecognised

data["unit_std"] = data[UNIT_COL].apply(norm_unit) if has_units else "ug/mL"   # tidy units; if no unit column, assume ug/mL
data["MW"] = data[SEQ_COL].apply(lambda s: peptides.Peptide(s).molecular_weight())   # molecular weight of each peptide

def to_ugml(row):                           # helper: convert one row's MIC to ug/mL
    if row["unit_std"] == "ug/mL":          # already in the right unit
        return row["MIC_value"]             # keep the value as it is
    if row["unit_std"] == "uM":             # micromolar needs converting
        return row["MIC_value"] * row["MW"] / 1000.0   # uM * g/mol / 1000 = ug/mL
    return np.nan                           # unit missing or unknown -> cannot convert

data["MIC_ugml"] = data.apply(to_ugml, axis=1)   # apply the conversion to every row (axis=1 = row by row)
dropped = int(data["MIC_ugml"].isna().sum())     # count rows that could not be converted
data = data[data["MIC_ugml"].notna()]       # drop those rows
print(f"Dropped {dropped} row(s) with missing/unconvertible units.")   # report how many were dropped
print("Remaining rows:", len(data))         # report how many rows remain

## Label & class balance

In [ ]:
data["label"] = (data["MIC_ugml"] <= MIC_THRESHOLD).astype(int)   # True/False -> 1 (active) / 0 (inactive)
counts = data["label"].value_counts().sort_index()   # count how many 0s and 1s, ordered 0 then 1
print("Class balance (0 = inactive, 1 = active):")   # heading for the counts
print(counts)                               # show the two counts
ratio = counts.max() / counts.min()         # how many times bigger the larger class is
imbalanced = ratio >= 2                     # rule of thumb: 2x or more = imbalanced (True/False)
print(f"Imbalance ratio: {ratio:.2f} -> {'imbalanced' if imbalanced else 'balanced'}")   # show ratio and verdict

## Features

In [ ]:
def features(seq):                          # turn ONE sequence into a dictionary of numbers
    p = peptides.Peptide(seq)               # wrap the sequence so we can ask it for properties
    f = {                                   # start the dictionary of features
        "length": len(seq),                 # number of amino acids
        "net_charge": p.charge(),           # overall electric charge
        "hydrophobicity": p.hydrophobicity(),   # average water-hating score
        "hydrophobic_moment": p.hydrophobic_moment() or 0.0,   # two-faced-ness; use 0.0 if none returned
        "isoelectric_point": p.isoelectric_point(),   # pH at which the net charge is zero
        "aliphatic_index": p.aliphatic_index(),       # share of small oily residues
    }
    for a in AA:                            # loop over each of the 20 amino acids
        f[f"pct_{a}"] = 100.0 * seq.count(a) / len(seq)   # percentage of the sequence that is amino acid a
    return f                                # hand back the finished dictionary

X = pd.DataFrame([features(s) for s in data[SEQ_COL]], index=data.index)   # features for every peptide -> table X (the inputs)
y = data["label"]                           # y = the answers (0/1 labels)
print("Feature matrix:", X.shape)           # show (peptides, features)
X.head()                                    # preview the first 5 rows

## Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(   # returns 4 pieces: train inputs/test inputs/train labels/test labels
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE   # 20% to test; keep class ratio equal in both; fixed seed
)
print("Train:", X_train.shape, " Test:", X_test.shape)   # show the size of each part

## Models

In [ ]:
cw = "balanced" if imbalanced else None     # class_weight: "balanced" gives the rarer class more attention
models = {                                  # dictionary: model name -> model object (not trained yet)
    "SVM": make_pipeline(StandardScaler(), SVC(class_weight=cw, random_state=RANDOM_STATE)),   # scale features, then SVM
    "Random Forest": RandomForestClassifier(class_weight=cw, random_state=RANDOM_STATE),       # many trees that vote
    "Gaussian NB": GaussianNB(),            # probability-based model (no class_weight option)
}

In [ ]:
results = {}                                # empty dictionary to store each model's scores
for name, model in models.items():          # go through the three models one at a time
    model.fit(X_train, y_train)             # fit = train: learn the pattern from training data + true labels
    pred = model.predict(X_test)            # predict = guess labels for the unseen test peptides
    results[name] = {                       # store this model's four scores
        "Accuracy": accuracy_score(y_test, pred),                       # share of all guesses that were right
        "Precision": precision_score(y_test, pred, zero_division=0),    # of predicted active, how many truly were
        "Recall": recall_score(y_test, pred, zero_division=0),          # of truly active, how many were found
        "F1": f1_score(y_test, pred, zero_division=0),                  # balance of precision and recall
    }
    print(f"\n=== {name} ===")               # heading with the model's name
    for metric, value in results[name].items():   # go through its four scores
        print(f"{metric:10s}: {value:.3f}")       # print name and score to 3 decimals
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["inactive", "active"])   # draw the 2x2 box of right/wrong guesses
    plt.title(name)                         # title the chart with the model's name
    plt.show()                              # display the chart

## Model comparison

In [ ]:
scores = pd.DataFrame(results).T[["Accuracy", "Precision", "Recall", "F1"]]   # results as a table: one row per model; fixed column order
scores.round(3)                             # show the table rounded to 3 decimals

## Random Forest feature importance

In [ ]:
rf = models["Random Forest"]                # take the already-trained Random Forest
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)   # importance per feature, biggest first, keep top 10
plt.figure(figsize=(7, 4))                  # start a chart 7 x 4 inches
sns.barplot(x=imp.values, y=imp.index, color="steelblue")   # horizontal bars: length = importance
plt.xlabel("Importance")                    # label the x axis
plt.title("Top 10 features (Random Forest)")   # chart title
plt.tight_layout()                          # tidy the spacing
plt.show()                                  # display the chart

## Does arrangement matter? Shuffled-sequence control

In [ ]:
best_name = scores["F1"].idxmax()           # name of the model with the highest F1
print("Best model by F1:", best_name)       # announce the winner

rng = np.random.default_rng(RANDOM_STATE)   # random-number generator with a fixed seed (reproducible)
def shuffle_seq(seq):                       # helper: scramble the order of one sequence
    chars = list(seq)                       # split the sequence into a list of letters
    rng.shuffle(chars)                      # shuffle the letters in place
    return "".join(chars)                   # glue them back into a string

shuffled = data[SEQ_COL].apply(shuffle_seq) # shuffle every sequence (same letters, new order)
X_shuf = pd.DataFrame([features(s) for s in shuffled], index=data.index)   # rebuild the features from shuffled sequences

Xs_train, Xs_test, ys_train, ys_test = train_test_split(   # split exactly as before so the comparison is fair
    X_shuf, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
m = clone(models[best_name])                # fresh untrained copy of the best model
m.fit(Xs_train, ys_train)                   # train it on the shuffled data
shuf_f1 = f1_score(ys_test, m.predict(Xs_test), zero_division=0)   # its F1 on the shuffled test set
orig_f1 = scores.loc[best_name, "F1"]       # the same model's original F1
print(f"Original F1 : {orig_f1:.3f}")       # F1 with real sequences
print(f"Shuffled F1 : {shuf_f1:.3f}")       # F1 with shuffled sequences
print(f"Drop        : {orig_f1 - shuf_f1:.3f}")   # how much the score fell